In [1]:
import os, sys, time
from dotenv import load_dotenv

load_dotenv("../.env")
sys.path.append("..")

from langchain_community.utilities import SQLDatabase

from src.chains import build_llm, SMALL_MODEL
from src.tools import search_video, search_wikipedia, search_technique

llm = build_llm()  # gpt-oss-120b
llm_small = build_llm(SMALL_MODEL)  # gpt-oss-20b: gerar SQL / reformular queries

# read-only na conexão = 1ª camada de segurança do text-to-SQL
db = SQLDatabase.from_uri("sqlite:///file:../data/db/bjj.sqlite?mode=ro&uri=true")
print(db.dialect, db.get_usable_table_names())
print(db.get_table_info())

print(
    db.run(
        "SELECT submission, COUNT(*) AS n FROM lutas "
        "WHERE submission IS NOT NULL GROUP BY submission ORDER BY n DESC LIMIT 5"
    )
)

try:
    db.run("DELETE FROM lutas WHERE 1=0")
except Exception as e:
    print("read-only OK:", type(e).__name__, str(e)[:80])

C:\Users\Giovanni Azevedo\AppData\Local\Temp\ipykernel_21220\4271154107.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


sqlite ['lutas']

CREATE TABLE lutas (
	victory_method TEXT, 
	submission TEXT, 
	winner_points INTEGER, 
	loser_points INTEGER, 
	female INTEGER, 
	year INTEGER, 
	absolute INTEGER, 
	weight_class REAL, 
	importance INTEGER, 
	total_points INTEGER, 
	submission_target TEXT, 
	winner_name TEXT, 
	loser_name TEXT
)

/*
3 rows from lutas table:
victory_method	submission	winner_points	loser_points	female	year	absolute	weight_class	importance	total_points	submission_target	winner_name	loser_name
DECISION	None	-1	-1	0	2011	1	None	1	2	None	Murilo Santana	Vinicius Magalhaes
SUBMISSION	Kimura	-1	-1	0	2022	0	3.0	0	2	Arm	Nicholas Meregali	Henrique Cardoso
DECISION	None	-1	-1	0	2022	0	3.0	1	2	None	Nicholas Meregali	Yuri Simoes
*/
[('RNC', 99), ('Armbar', 68), ('Guillotine', 34), ('Heel hook', 32), ('Triangle', 24)]
read-only OK: OperationalError (sqlite3.OperationalError) attempt to write a readonly database
[SQL: DELETE FRO


In [2]:
import importlib, src.tools
importlib.reload(src.tools)          # src.tools já foi importado na célula 1
from src.tools import ibjjf_points

print(ibjjf_points.invoke("Guard pass"))
print(ibjjf_points.invoke("berimbolo"))
print(ibjjf_points.name, "|", ibjjf_points.args)


guard pass: 3 points (IBJJF)
Unknown action 'berimbolo'. Valid: takedown, sweep, knee on belly, guard pass, mount, back control
ibjjf_points | {'action': {'title': 'Action', 'type': 'string'}}


## 4.1 Tipologia das ferramentas

| Tipo            | Tool                               | Quando usar                          | Custo                 | Determinismo                      | Risco                                             |
| --------------- | ---------------------------------- | ------------------------------------ | --------------------- | --------------------------------- | ------------------------------------------------- |
| Hardcoded       | `ibjjf_points`                     | regra fixa e pequena (pontuação)     | zero, sem rede        | total                             | desatualizar se a regra mudar                     |
| Semântica (RAG) | `search_technique`                 | "como fazer/defender X" na KB curada | embedding local (CPU) | alto (mesmo índice → mesmo top-k) | chunk errado se o vocabulário não casar (ver 4.5) |
| API externa     | `search_video`, `search_wikipedia` | vídeo / fato histórico fora da KB    | rede + quota          | baixo (resultado muda no tempo)   | timeout, quota, conteúdo não curado               |
| Text-to-query   | `query_adcc` (4.3)                 | estatística sobre dados estruturados | 1 chamada LLM + SQL   | médio (LLM gera o SQL)            | SQL errado ou perigoso → read-only + validação    |


In [3]:
importlib.reload(src.tools)
from src.tools import search_wikipedia, search_video, _wiki_lookup

for i in range(2):
    t0 = time.perf_counter()
    out = search_wikipedia.invoke("Helio Gracie")
    print(f"call {i+1}: {time.perf_counter()-t0:.3f}s | {out[:60]}")

print(_wiki_lookup.cache_info())
print(search_video.invoke("kimura closed guard bjj"))


call 1: 0.678s | Hélio Gracie: Hélio Gracie was a Brazilian martial artist wh
call 2: 0.002s | Hélio Gracie: Hélio Gracie was a Brazilian martial artist wh
CacheInfo(hits=1, misses=1, maxsize=128, currsize=1)
Kimura Tutorial — https://www.youtube.com/watch?v=fqQ4mVxJaoE


## 4.2 Integração de APIs por tipo de uso

| Uso | Tool | Fonte | Timeout | Erro | Retry | Cache | Dummy |
|---|---|---|---|---|---|---|---|
| Busca (descoberta) | `search_video` | YouTube Data API v3 (quota 10k/dia, 100 por busca) | 10s | string → agente segue | 2x em 429/5xx | `lru_cache` 128 | sim |
| Consulta (lookup factual) | `search_wikipedia` | Wikipedia REST (sem chave, exige User-Agent) | 10s | string | 2x em 429/5xx | `lru_cache` 128 | sim |
| Local (conhecimento curado) | `search_technique` | FAISS em disco | — | string | — | índice carregado 1x | sim |
| Local (dados estruturados) | `query_adcc` | SQLite read-only | — | string + retry c/ erro no prompt (4.3) | — | — | sim |
| Ação/escrita | nenhuma | — | — | — | — | — | — |

Sem tools de escrita por decisão: o agente é tutor, não executa ações no mundo (agendar, postar, pagar). Isso elimina a necessidade de confirmação humana / idempotência nesta fase.


## 4.3 Text-to-SQL sobre SQLite

Pergunta em linguagem natural → LLM gera SQL estruturado (`SQLQuery`) → validação → execução read-only → resposta.
Segurança em 2 camadas: (1) conexão `mode=ro` (provada na célula 1); (2) validador que só aceita um único SELECT.


In [4]:
import re
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser


class SQLQuery(BaseModel):
    reasoning: str = Field(
        description="Short reasoning: which columns and filters answer the question"
    )
    sql: str = Field(description="One SQLite SELECT query")


sql_parser = PydanticOutputParser(pydantic_object=SQLQuery)

COLUMN_DICT = """Column notes (table lutas = one ADCC match per row, 1998-2022):
- victory_method: 'SUBMISSION', 'POINTS', 'DECISION', 'INJURY', 'DESQUALIFICATION' (sic, misspelled)
- submission: free text, only when victory_method='SUBMISSION', else NULL. Ex: 'RNC' (rear naked choke),
  'Armbar', 'Guillotine', 'Heel hook', 'Inside heel hook', 'Outside heel hook', 'Triangle', 'Kimura'.
  Use LIKE '%heel hook%' to get all heel hook variants.
- submission_target: 'Arm', 'Leg', 'Neck', 'Other/Unknown', or NULL (not a submission)
- winner_points / loser_points: -1 means NO DATA (exclude with > -1 before averaging)
- female: 1 = women's division, 0 = men's
- absolute: 1 = open weight (absolute) division; then weight_class is NULL
- weight_class: code 0-4 (lightest to heaviest), NULL when absolute
- importance: round importance code 0-5 (higher = later round)
- winner_name / loser_name: athlete full name"""

FEW_SHOT = """Examples:
Q: Most common submission ever?
SQL: SELECT submission, COUNT(*) AS n FROM lutas WHERE submission IS NOT NULL GROUP BY submission ORDER BY n DESC LIMIT 1
Q: How many women's matches ended by submission?
SQL: SELECT COUNT(*) FROM lutas WHERE female = 1 AND victory_method = 'SUBMISSION' LIMIT 1
Q: Average winner points in 2019, when available?
SQL: SELECT AVG(winner_points) FROM lutas WHERE year = 2019 AND winner_points > -1 LIMIT 1"""

sql_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You write SQLite queries for an ADCC (submission grappling) match database.\n"
            "Schema:\n{schema}\n\n{column_dict}\n\n{few_shot}\n\n"
            "Rules: a single SELECT only (no ;, no writes). Always end the query with a LIMIT clause "
            "(LIMIT 20, or LIMIT 1 for a single aggregate). "
            "Use only the columns above. Athlete names: use LIKE '%name%'.\n\n{format_instructions}",
        ),
        ("human", "{question}"),
    ]
).partial(
    schema=db.get_table_info(),
    column_dict=COLUMN_DICT,
    few_shot=FEW_SHOT,
    format_instructions=sql_parser.get_format_instructions(),
)

sql_gen_chain = sql_prompt | llm_small | sql_parser

FORBIDDEN = re.compile(
    r"\b(insert|update|delete|drop|alter|create|attach|detach|pragma|vacuum)\b", re.I
)


def validate_sql(sql: str) -> str:
    """2ª camada de segurança: só um SELECT, sem comandos de escrita."""
    s = sql.strip().rstrip(";").strip()
    if not s.lower().startswith(("select", "with")):
        raise ValueError("only SELECT queries are allowed")
    if ";" in s:
        raise ValueError("multiple statements are not allowed")
    if FORBIDDEN.search(s):
        raise ValueError(f"forbidden keyword: {FORBIDDEN.search(s).group()}")
    return s


# guarda: casos maliciosos (sem LLM)
for bad in [
    "DROP TABLE lutas",
    "SELECT 1; DELETE FROM lutas",
    "PRAGMA table_info(lutas)",
]:
    try:
        validate_sql(bad)
    except ValueError as e:
        print(f"BLOCKED  {bad!r:35} -> {e}")

# geração: 3 perguntas
for q in [
    "Quantas lutas terminaram em finalização em 2022?",
    "Qual atleta tem mais vitórias?",
    "Quantas vitórias por heel hook existem, somando todas as variações?",
]:
    out = sql_gen_chain.invoke({"question": q})
    print(
        f"\n>>> {q}\n    reasoning: {out.reasoning}\n    sql: {validate_sql(out.sql)}"
    )
    time.sleep(5)

BLOCKED  'DROP TABLE lutas'                  -> only SELECT queries are allowed
BLOCKED  'SELECT 1; DELETE FROM lutas'       -> multiple statements are not allowed
BLOCKED  'PRAGMA table_info(lutas)'          -> only SELECT queries are allowed

>>> Quantas lutas terminaram em finalização em 2022?
    reasoning: Use the victory_method column to filter for 'SUBMISSION' and the year column to restrict to 2022, then count the matching rows.
    sql: SELECT COUNT(*) FROM lutas WHERE year = 2022 AND victory_method = 'SUBMISSION' LIMIT 1

>>> Qual atleta tem mais vitórias?
    reasoning: To find the athlete with the most victories, we count how many times each name appears in the winner_name column, group by that name, order by the count descending, and return the top result.
    sql: SELECT winner_name, COUNT(*) AS n FROM lutas GROUP BY winner_name ORDER BY n DESC LIMIT 1

>>> Quantas vitórias por heel hook existem, somando todas as variações?
    reasoning: We need to count all matches wher

In [5]:
import sqlite3
from langchain_core.output_parsers import StrOutputParser


def run_sql_question(question: str, max_attempts: int = 2) -> dict:
    """Gera SQL, valida e executa. Se falhar, devolve o erro ao LLM e tenta de novo."""
    prompt_q, sql, error = question, "", ""
    for attempt in range(1, max_attempts + 1):
        out = sql_gen_chain.invoke({"question": prompt_q})
        sql = out.sql
        try:
            rows = db.run(validate_sql(sql))
            return {"sql": sql, "rows": rows, "attempts": attempt, "error": None}
        except Exception as e:
            error = str(e).split("\n")[0][:200]  # só a 1ª linha do erro
            prompt_q = (
                f"{question}\n\nYour previous query failed.\n"
                f"Query: {sql}\nError: {error}\nWrite a corrected query."
            )
    return {"sql": sql, "rows": None, "attempts": attempt, "error": error}


answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Você responde perguntas sobre estatísticas do ADCC. Use SOMENTE o resultado SQL abaixo. "
            "Responda em português, em 1-2 frases. Se o resultado estiver vazio, diga que não há dados.\n\n"
            "SQL: {sql}\nResultado: {rows}",
        ),
        ("human", "{question}"),
    ]
)
answer_chain = answer_prompt | llm_small | StrOutputParser()


# gabarito calculado direto no SQLite (sem LLM)
raw = sqlite3.connect("../data/db/bjj.sqlite")
gold_sql = {
    "Quantas lutas terminaram em finalização em 2022?": "SELECT COUNT(*) FROM lutas WHERE year=2022 AND victory_method='SUBMISSION'",
    "Qual atleta tem mais vitórias?": "SELECT winner_name FROM lutas GROUP BY winner_name ORDER BY COUNT(*) DESC LIMIT 1",
    "Quantas vitórias por heel hook existem, somando todas as variações?": "SELECT COUNT(*) FROM lutas WHERE submission LIKE '%heel hook%'",
    "Qual a finalização mais comum da história do ADCC?": "SELECT submission FROM lutas WHERE submission IS NOT NULL GROUP BY submission ORDER BY COUNT(*) DESC LIMIT 1",
    "Quantas vitórias o Marcelo Garcia tem?": "SELECT COUNT(*) FROM lutas WHERE winner_name LIKE '%Marcelo Garcia%'",
    "Qual a média de pontos do vencedor, considerando só lutas com placar registrado?": "SELECT ROUND(AVG(winner_points), 2) FROM lutas WHERE winner_points > -1",
}


def matches(gold: str, rows) -> bool:
    """Número: compara com tolerância. Texto: substring."""
    if rows is None:
        return False
    try:
        g = float(gold)
    except ValueError:
        return gold in rows
    nums = [float(n) for n in re.findall(r"-?\d+(?:\.\d+)?", rows)]
    return any(abs(n - g) < 0.01 for n in nums)


results = []
for q, gsql in gold_sql.items():
    gold = str(raw.execute(gsql).fetchone()[0])
    r = run_sql_question(q)
    ok = matches(gold, r["rows"])
    results.append(
        {
            "question": q,
            "gold": gold,
            "ok": ok,
            "attempts": r["attempts"],
            "sql": r["sql"],
        }
    )
    print(
        f"{'OK ' if ok else 'ERR'} [{r['attempts']}x] {q}\n     gold={gold} | rows={r['rows']} | err={r['error']}"
    )
    time.sleep(5)

print(f"\nAcerto: {sum(x['ok'] for x in results)}/{len(results)}")

# resposta final em linguagem natural (1 exemplo)
r = run_sql_question("Quantas vitórias o Marcelo Garcia tem?")
print(
    "\n",
    answer_chain.invoke({"question": "Quantas vitórias o Marcelo Garcia tem?", **r}),
)

OK  [1x] Quantas lutas terminaram em finalização em 2022?
     gold=40 | rows=[(40,)] | err=None
OK  [1x] Qual atleta tem mais vitórias?
     gold=Alexandre Ribeiro | rows=[('Alexandre Ribeiro', 30)] | err=None
OK  [1x] Quantas vitórias por heel hook existem, somando todas as variações?
     gold=52 | rows=[(52,)] | err=None
OK  [1x] Qual a finalização mais comum da história do ADCC?
     gold=RNC | rows=[('RNC', 99)] | err=None
OK  [1x] Quantas vitórias o Marcelo Garcia tem?
     gold=27 | rows=[(27,)] | err=None
OK  [1x] Qual a média de pontos do vencedor, considerando só lutas com placar registrado?
     gold=4.05 | rows=[(4.048913043478261,)] | err=None

Acerto: 6/6

 Marcelo Garcia tem 27 vitórias.


In [6]:
importlib.reload(src.tools)
from src.tools import query_adcc

print(query_adcc.invoke("Qual a finalização mais comum entre as mulheres?"))
time.sleep(5)
print(query_adcc.invoke("Quem venceu mais lutas por finalização no pescoço?"))

src.tools.USE_DUMMY = True               # simula ambiente do professor sem chave
print(query_adcc.invoke("qualquer coisa"))
src.tools.USE_DUMMY = False


As finalizações mais comuns entre as mulheres são **Toe hold** e **RNC**, cada uma ocorrendo 3 vezes.
SQL: SELECT submission, COUNT(*) AS n FROM lutas WHERE female = 1 AND victory_method = 'SUBMISSION' GROUP BY submission ORDER BY n DESC LIMIT 5
Marcelo Garcia venceu mais lutas por finalização no pescoço, com 19 vitórias.
SQL: SELECT winner_name, COUNT(*) AS n FROM lutas WHERE victory_method='SUBMISSION' AND submission_target='Neck' GROUP BY winner_name ORDER BY n DESC LIMIT 5
(dummy) RNC is the most common submission in ADCC (99 wins).
SQL: SELECT submission, COUNT(*) ...
